# Nyaya-Sahayak — Evaluation Notebook

Runs all benchmark triage questions through the pipeline and computes:

- **Domain classification accuracy** (exact match)
- **Section citation recall** (expected sections found in response)
- **Action plan completeness** (helplines present, filing body, steps)
- **Per-category breakdown**

Results are displayed as tables and logged to MLflow (if available).

In [0]:
# %pip install -q mlflow
# print("✅ dependencies ready")

In [0]:
import json, os, sys, re, time
from collections import defaultdict

REPO_ROOT = "/Workspace/Users/saisandeshk@iisc.ac.in/bharat-bricks-hacks/"
_src = os.path.join(REPO_ROOT, "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

## 1. Load benchmark

In [0]:
BENCHMARK_PATH = os.path.join(REPO_ROOT, "tests", "benchmark_questions.json")
with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

triage_qs = benchmark.get("triage_questions", [])
print(f"📋 {len(triage_qs)} triage questions loaded")
for q in triage_qs[:3]:
    print(f"  {q['id']}: {q['question'][:60]}...")

## 2. Run triage pipeline on each question

In [0]:
import os

# LLM config (MANDATORY)
os.environ["LLM_OPENAI_BASE_URL"] = "https://7474650313055161.ai-gateway.cloud.databricks.com/mlflow/v1"
os.environ["LLM_MODEL"] = "databricks-llama-4-maverick"

# OPTIONAL: Let SDK handle auth
# os.environ["DATABRICKS_TOKEN"] = ""

# Sarvam (optional)
try:
    os.environ["SARVAM_API_KEY"] = dbutils.secrets.get("nyaya-dhwani", "sarvam_api_key")
except Exception:
    print("⚠️ SARVAM not configured")

# HF (optional)
try:
    os.environ["HF_TOKEN"] = dbutils.secrets.get("nyaya-dhwani", "hf_token")
except Exception:
    print("⚠️ HF_TOKEN not found")

In [0]:
import os

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("User:", w.current_user.me().user_name)

print("LLM_OPENAI_BASE_URL:", os.environ.get("LLM_OPENAI_BASE_URL"))
print("LLM_CHAT_COMPLETIONS_URL:", os.environ.get("LLM_CHAT_COMPLETIONS_URL"))
print("LLM_MODEL:", os.environ.get("LLM_MODEL"))
print("DATABRICKS_TOKEN:", "SET" if os.environ.get("DATABRICKS_TOKEN") else "NOT SET")
print("OPENAI_API_KEY:", "SET" if os.environ.get("OPENAI_API_KEY") else "NOT SET")
print("LLM_API_KEY:", "SET" if os.environ.get("LLM_API_KEY") else "NOT SET")

In [0]:
# Install required dependencies
%pip install -q 'faiss-cpu>=1.8.0' 'sentence-transformers'

# Add correct path for nyaya_dhwani imports
src_path = '/Workspace/Users/saisandeshk@iisc.ac.in/bharat-bricks-hacks/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Clear any cached imports to force fresh import
if 'nyaya_dhwani' in sys.modules:
    del sys.modules['nyaya_dhwani']
if 'nyaya_dhwani.domain_classifier' in sys.modules:
    del sys.modules['nyaya_dhwani.domain_classifier']
if 'nyaya_dhwani.retriever' in sys.modules:
    del sys.modules['nyaya_dhwani.retriever']
if 'nyaya_dhwani.triage_engine' in sys.modules:
    del sys.modules['nyaya_dhwani.triage_engine']

from nyaya_dhwani.domain_classifier import classify_domain
from nyaya_dhwani.retriever import get_retriever
from nyaya_dhwani.triage_engine import build_triage_context
from nyaya_dhwani.llm_client import chat_completions, extract_assistant_text
from nyaya_dhwani.triage_engine import TRIAGE_SYSTEM_PROMPT

# Translation for non-English queries (same as the live app does)
try:
    from nyaya_dhwani.sarvam_client import translate_text
    _HAS_TRANSLATE = True
    print("✅ Sarvam translation available")
except Exception:
    _HAS_TRANSLATE = False
    print("⚠️ Sarvam translation unavailable — non-English queries will be tested as-is")

retriever = get_retriever()

results = []

for idx, q in enumerate(triage_qs):
    qid = q["id"]
    query = q["question"]
    lang = q.get("language", "en")

    expected_domain = q.get("expected_domain", "")
    expected_sections = q.get("expected_sections", [])
    expected_action_steps = q.get("expected_action_steps", [])
    expected_helplines = q.get("expected_helplines", [])

    t0 = time.perf_counter()

    # Step 0: Translate non-English queries to English (mirrors live app pipeline)
    query_en = query
    if lang != "en" and _HAS_TRANSLATE:
        try:
            query_en = translate_text(query, source_language_code="auto", target_language_code="en-IN")
            print(f"  [{qid}] Translated ({lang}→en): {query_en[:80]}...")
        except Exception as e:
            print(f"  [{qid}] Translation failed: {e}")

    # Step 1: Domain classification (on English text, as the live app does)
    domains = classify_domain(query_en)
    detected_domain = domains[0].domain if domains else "unknown"
    detected_situation = domains[0].situation_type if domains else "unknown"

    # Step 2: Retrieve chunks (using English query)
    chunks_df = retriever.search(query_en, k=7)

    # Step 3: Build triage context + action plan
    triage_domains, action_plan, enriched_msg = build_triage_context(query_en, chunks_df)

    # Step 4: Generate LLM response
    try:
        messages = [
            {"role": "system", "content": TRIAGE_SYSTEM_PROMPT},
            {"role": "user", "content": enriched_msg}
        ]
        response = chat_completions(messages)
        llm_response = extract_assistant_text(response)
    except Exception as e:
        llm_response = f"[LLM error: {e}]"
    print("====== LLM RESPONSE ======")
    print(llm_response[:500])
    elapsed = time.perf_counter() - t0

    # --- Score: domain accuracy ---
    domain_correct = detected_domain == expected_domain

    # --- Score: section citation recall ---
    response_lower = llm_response.lower()
    sections_hit = sum(1 for s in expected_sections if s.lower() in response_lower)
    section_recall = sections_hit / max(len(expected_sections), 1)

    # --- Score: action plan completeness ---
    steps_hit = sum(1 for s in expected_action_steps if s.lower() in response_lower)
    steps_recall = steps_hit / max(len(expected_action_steps), 1)

    helplines_hit = 0
    if action_plan and action_plan.helplines:
        for hl in expected_helplines:
            for ap_hl in action_plan.helplines:
                if hl in ap_hl.get("number", ""):
                    helplines_hit += 1
                    break
    helpline_recall = helplines_hit / max(len(expected_helplines), 1) if expected_helplines else 1.0

    has_filing_body = bool(action_plan and action_plan.filing_body)
    has_fee_info = bool(action_plan and action_plan.filing_fee)

    result = {
        "id": qid,
        "category": q.get("category", ""),
        "language": q.get("language", "en"),
        "query": query[:80],
        "expected_domain": expected_domain,
        "detected_domain": detected_domain,
        "detected_situation": detected_situation,
        "domain_correct": domain_correct,
        "section_recall": round(section_recall, 2),
        "steps_recall": round(steps_recall, 2),
        "helpline_recall": round(helpline_recall, 2),
        "has_filing_body": has_filing_body,
        "has_fee_info": has_fee_info,
        "response_time_s": round(elapsed, 2),
        "response_preview": llm_response[:200],
    }
    results.append(result)

    print(f"[{idx+1}/{len(triage_qs)}] {qid}: domain={'✅' if domain_correct else '❌'} "
          f"sections={section_recall:.0%} steps={steps_recall:.0%} ({elapsed:.1f}s)")

## 3. Per-question results table

In [0]:
import pandas as pd
results_df = pd.DataFrame(results)

results_df[[
    "id", "category", "expected_domain", "detected_domain",
    "domain_correct", "section_recall", "steps_recall",
    "helpline_recall", "has_filing_body", "response_time_s"
]]

## 4. Aggregate metrics

In [0]:
n = len(results_df)
domain_acc = results_df["domain_correct"].mean()
avg_section_recall = results_df["section_recall"].mean()
avg_steps_recall = results_df["steps_recall"].mean()
avg_helpline_recall = results_df["helpline_recall"].mean()
filing_body_pct = results_df["has_filing_body"].mean()
fee_info_pct = results_df["has_fee_info"].mean()
avg_time = results_df["response_time_s"].mean()

print(f"""
{'='*55}
📊 NYAYA-SAHAYAK EVALUATION (n={n})
{'='*55}
Domain classification accuracy : {domain_acc:.1%}
Avg section citation recall    : {avg_section_recall:.1%}
Avg action-step recall         : {avg_steps_recall:.1%}
Avg helpline recall            : {avg_helpline_recall:.1%}
Filing body present            : {filing_body_pct:.1%}
Fee information present        : {fee_info_pct:.1%}
Avg response time              : {avg_time:.1f}s
{'='*55}
""")

## 5. Per-category breakdown

In [0]:
cat_df = results_df.groupby("category").agg(
    count=("id", "count"),
    domain_accuracy=("domain_correct", "mean"),
    avg_section_recall=("section_recall", "mean"),
    avg_steps_recall=("steps_recall", "mean"),
    avg_time=("response_time_s", "mean"),
).round(2)

cat_df

## 6. Log to MLflow

In [0]:
import mlflow

EXPERIMENT_NAME = "/Users/saisandeshk@iisc.ac.in/nyaya-sahayak-rag-evaluation"
mlflow.set_experiment(EXPERIMENT_NAME)

PHASE = os.environ.get("NYAYA_PHASE", "phase-2")

with mlflow.start_run(run_name=f"full-eval-{PHASE}") as run:
    mlflow.set_tag("phase", PHASE)
    mlflow.set_tag("evaluator", "evaluation_notebook")
    mlflow.set_tag("benchmark_size", str(n))

    mlflow.log_param("embedding_model", "sentence-transformers/all-MiniLM-L6-v2")
    mlflow.log_param("retrieval_backend", os.environ.get("NYAYA_RETRIEVAL_BACKEND", "hybrid_faiss_bm25"))
    mlflow.log_param("llm_model", os.environ.get("LLM_MODEL", "databricks-llama-4-maverick"))
    mlflow.log_param("num_questions", n)
    mlflow.log_param("top_k", 7)

    mlflow.log_metric("domain_accuracy", domain_acc)
    mlflow.log_metric("avg_section_recall", avg_section_recall)
    mlflow.log_metric("avg_steps_recall", avg_steps_recall)
    mlflow.log_metric("avg_helpline_recall", avg_helpline_recall)
    mlflow.log_metric("filing_body_pct", filing_body_pct)
    mlflow.log_metric("fee_info_pct", fee_info_pct)
    mlflow.log_metric("avg_response_time_s", avg_time)

    results_path = "/tmp/eval_detailed_results.json"
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2, default=str)

    mlflow.log_artifact(results_path)

    print(f"✅ MLflow run: {run.info.run_id}")

## 7. Error analysis — incorrect domain classifications

In [0]:
wrong = results_df[~results_df["domain_correct"]]

if wrong.empty:
    print("🎉 All domain classifications are correct!")
else:
    print(f"❌ {len(wrong)} incorrect classifications:")
    wrong[["id", "query", "expected_domain", "detected_domain", "detected_situation"]]

## 8. Low section recall analysis

In [0]:
low_recall = results_df[results_df["section_recall"] < 0.4]

if low_recall.empty:
    print("🎉 All questions have ≥50% section recall!")
else:
    print(f"⚠️ {len(low_recall)} questions with <50% section recall:")
    low_recall[["id", "query", "section_recall", "response_preview"]]